# Notebook 3 — Adaptive Execution Simulation

**Project:** Adaptive Execution via Online Parameter Estimation  
**Author:** Changkui Wu (FSU Financial Mathematics PhD, 2026)

---

## Overview

This notebook is the centrepiece of the project.  We simulate the
**closed-loop adaptive execution system** studied in the thesis:

$$\underbrace{dD_t = -r^* D_t \, dt - k \, m_t \, dt + \sigma \, dB_t}_{\text{OW LOB dynamics (environment)}}$$

$$\underbrace{m_t = \frac{\hat{r}_{t-} X_t}{\hat{r}_{t-}(T-t) + 2}}_{\text{Adaptive OW controller (Proposition 3)}}$$

$$\underbrace{d\hat{r}_t = \eta_t (-D_t) \sigma^{-2} \left[\frac{dD_t}{dt} + \hat{r}_t D_t\right] dt + \cdots}_{\text{Online estimator (Thesis Eq. 3.2)}}$$

The three equations are **genuinely coupled**:
$\hat{r}_t$ drives $m_t$, which drives $D_t$, which drives $\hat{r}_{t+dt}$.
This is the closed-loop adaptive system whose almost-sure convergence is
proved in the thesis — and whose cost performance we evaluate here.

### Strategies compared

| Strategy | Description |
|----------|-------------|
| **TWAP** | Baseline: split order evenly over horizon |
| **Static OW** | OW Proposition 3 with a *fixed* (possibly wrong) $r$ |
| **Adaptive OW** | OW Proposition 3 with *online-estimated* $\hat{r}_t$ |

The key question: **when $r$ is unknown, does the adaptive strategy
outperform a static strategy with a misspecified $r$?**


In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from lobster import load_lobster, estimate_params
from ow_model import OWParams, OWEnvironment, monte_carlo

DARK, GRID = '#0f1117', '#1a1d2e'
TEXT, BLUE  = '#e0e0e0', '#4a9eff'
GREEN, ORANGE, RED = '#50fa7b', '#ffb86c', '#ff5555'
PURPLE, CYAN, YELLOW = '#bd93f9', '#8be9fd', '#f1fa8c'

plt.rcParams.update({
    'figure.facecolor': DARK, 'axes.facecolor': GRID,
    'axes.edgecolor': '#333355', 'axes.labelcolor': TEXT,
    'xtick.color': TEXT, 'ytick.color': TEXT,
    'text.color': TEXT, 'grid.color': '#252540',
    'grid.linewidth': 0.6, 'legend.facecolor': GRID,
    'legend.labelcolor': TEXT, 'font.size': 10,
})

# ── Calibrated parameters from Notebook 1 ────────────────────────────────
DATA_DIR = os.path.join('..', 'data')
lob      = load_lobster(
    os.path.join(DATA_DIR, 'AAPL_2012-06-21_34200000_57600000_message_5.csv'),
    os.path.join(DATA_DIR, 'AAPL_2012-06-21_34200000_57600000_orderbook_5.csv'),
)
cal = estimate_params(lob)

params = OWParams(
    r     = cal['r'],      # TRUE resilience (used by the environment)
    sigma = cal['sigma'],
    q     = cal['q'],
    lam   = cal['lam'],
    k     = cal['k'],
    s     = cal['spread_mean'],
    F0    = float(lob.mid.mean()),
)

print("Calibrated OW parameters (from real LOBSTER data):")
for k, v in vars(params).items():
    print(f"  {k:6s} = {v:.5f}")
print()
print("Simulation settings:")
X0, T, dt = 5000, 390, 1.0
print(f"  Order size X0 = {X0} shares")
print(f"  Horizon     T = {T} seconds ({T/60:.1f} minutes)")
print(f"  Time step  dt = {dt} second")


## 1. Single-Path Visualisation

A single Monte Carlo path showing how the three strategies execute the order
and how the adaptive estimator's $\hat{r}_t$ evolves over time.


In [ ]:
env  = OWEnvironment(params, X0=X0, T=T, dt=dt, seed=7)
r_tw = env.run_twap()
r_st = env.run_static_ow()                  # uses true r (oracle)
r_ad = env.run_adaptive_ow(r0=2.0,          # starts with wrong r
                            eta0=0.1, alpha=0.8)

t_plot = r_tw.t

fig = plt.figure(figsize=(15, 12))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)
fig.suptitle('Single-Path Adaptive Execution Simulation', color=TEXT, fontsize=13)

# ── Panel 1: Remaining order Xt ───────────────────────────────────────────
ax = fig.add_subplot(gs[0, :])
ax.plot(t_plot, r_tw.Xt, color=RED,    lw=1.5, label='TWAP')
ax.plot(t_plot, r_st.Xt, color=YELLOW, lw=1.5, label=f'Static OW (r={params.r:.3f})')
ax.plot(t_plot, r_ad.Xt, color=GREEN,  lw=1.5, label='Adaptive OW (r₀=2.0)')
ax.set_ylabel('Remaining order $X_t$ (shares)')
ax.set_xlabel('Time (seconds)')
ax.set_title('Execution Profile — Remaining Order')
ax.legend(fontsize=9)
ax.set_facecolor(GRID)

# ── Panel 2: Trading rate mt ──────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 0])
ax.plot(t_plot, r_tw.mt, color=RED,    lw=1.0, alpha=0.8, label='TWAP')
ax.plot(t_plot, r_st.mt, color=YELLOW, lw=1.0, alpha=0.8, label='Static OW')
ax.plot(t_plot, r_ad.mt, color=GREEN,  lw=1.0, alpha=0.8, label='Adaptive OW')
ax.set_ylabel('Trading rate $m_t$ (shares/s)')
ax.set_xlabel('Time (seconds)')
ax.set_title('Trading Rate')
ax.legend(fontsize=9)
ax.set_facecolor(GRID)

# ── Panel 3: LOB deviation Dt ─────────────────────────────────────────────
ax = fig.add_subplot(gs[1, 1])
ax.plot(t_plot, r_tw.Dt, color=RED,    lw=0.8, alpha=0.8, label='TWAP')
ax.plot(t_plot, r_st.Dt, color=YELLOW, lw=0.8, alpha=0.8, label='Static OW')
ax.plot(t_plot, r_ad.Dt, color=GREEN,  lw=0.8, alpha=0.8, label='Adaptive OW')
ax.axhline(0, color=TEXT, lw=0.6, ls='--', alpha=0.4)
ax.set_ylabel('LOB deviation $D_t$ ($)')
ax.set_xlabel('Time (seconds)')
ax.set_title('LOB Deviation Path')
ax.legend(fontsize=9)
ax.set_facecolor(GRID)

# ── Panel 4: r_hat convergence ────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 0])
ax.plot(t_plot, r_ad.r_hat, color=CYAN, lw=1.5, label='$\hat{r}_t$ (adaptive)')
ax.axhline(params.r, color=RED, lw=1.5, ls='--',
           label=f'$r^*$ = {params.r:.4f} (true)')
ax.axhline(2.0, color=ORANGE, lw=1.0, ls=':', alpha=0.7, label='$r_0$ = 2.0 (initial)')
ax.set_ylabel('$\hat{r}_t$ (per second)')
ax.set_xlabel('Time (seconds)')
ax.set_title('Online Estimator: $\hat{r}_t$ Convergence
(Theorem 4.3: a.s. convergence)')
ax.legend(fontsize=9)
ax.set_facecolor(GRID)
ax.set_ylim(0, max(2.5, r_ad.r_hat.max() * 1.1))

# ── Panel 5: Cumulative cost ──────────────────────────────────────────────
ax = fig.add_subplot(gs[2, 1])
# Approximate cumulative cost (mid-quote * trade rate)
cum_tw = np.cumsum(r_tw.Vt * r_tw.mt * dt)
cum_st = np.cumsum(r_st.Vt * r_st.mt * dt)
cum_ad = np.cumsum(r_ad.Vt * r_ad.mt * dt)
ax.plot(t_plot[:-1], cum_tw, color=RED,    lw=1.2, label=f'TWAP      ${r_tw.cost:,.0f}')
ax.plot(t_plot[:-1], cum_st, color=YELLOW, lw=1.2, label=f'Static OW ${r_st.cost:,.0f}')
ax.plot(t_plot[:-1], cum_ad, color=GREEN,  lw=1.2, label=f'Adaptive  ${r_ad.cost:,.0f}')
ax.set_ylabel('Cumulative cost ($)')
ax.set_xlabel('Time (seconds)')
ax.set_title('Cumulative Execution Cost')
ax.legend(fontsize=8)
ax.set_facecolor(GRID)

plt.savefig(os.path.join('..','outputs','nb3_single_path.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()

print(f"Single-path costs:")
print(f"  TWAP:       ${r_tw.cost:,.2f}")
print(f"  Static OW:  ${r_st.cost:,.2f}  (savings ${r_tw.cost-r_st.cost:+,.2f} vs TWAP)")
print(f"  Adaptive:   ${r_ad.cost:,.2f}  (savings ${r_tw.cost-r_ad.cost:+,.2f} vs TWAP)")


## 2. The Key Experiment: Robustness to Misspecification

**The realistic scenario:** the practitioner does not know $r^*$.
They have a prior guess $r_0$.

- **Static OW** uses $r_0$ and *never updates* — its strategy is sub-optimal
  if $r_0 \neq r^*$.
- **Adaptive OW** starts from $r_0$ but *learns* $r^*$ over time.

We sweep over different initial guesses and compare costs over 300 Monte Carlo paths.


In [ ]:
n_mc   = 300
r_vals = [0.005, 0.01, 0.02, params.r, 0.1, 0.2, 0.5, 1.0]

costs_tw_all  = []
costs_st_dict = {r: [] for r in r_vals}
costs_ad_dict = {r: [] for r in r_vals}

print("Running Monte Carlo (this takes ~1 min)...")
for seed in range(n_mc):
    env = OWEnvironment(params, X0=X0, T=T, dt=dt, seed=seed)
    costs_tw_all.append(env.run_twap().cost)
    for r0 in r_vals:
        env2 = OWEnvironment(params, X0=X0, T=T, dt=dt, seed=seed)
        costs_st_dict[r0].append(env2.run_static_ow(r=r0).cost)
        env3 = OWEnvironment(params, X0=X0, T=T, dt=dt, seed=seed)
        costs_ad_dict[r0].append(env3.run_adaptive_ow(r0=r0, eta0=0.1, alpha=0.8).cost)
    if (seed+1) % 50 == 0:
        print(f"  {seed+1}/{n_mc} paths done")

ct = np.mean(costs_tw_all)
print(f"\nTWAP mean cost: ${ct:,.2f}")


In [ ]:
# ── Plot: savings vs TWAP as function of r0 ──────────────────────────────
r_arr    = np.array(r_vals)
save_st  = np.array([(ct - np.mean(costs_st_dict[r]))/ct*100 for r in r_vals])
save_ad  = np.array([(ct - np.mean(costs_ad_dict[r]))/ct*100 for r in r_vals])
ad_beats = save_ad - save_st   # positive = adaptive beats static

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'Robustness to Misspecification  |  {n_mc} MC paths  '
    f'|  True $r^*$={params.r:.4f}',
    color=TEXT, fontsize=12)

# Panel 1: absolute savings
ax = axes[0]
ax.semilogx(r_arr, save_st, color=YELLOW, lw=2.0, marker='s', ms=6,
            label='Static OW')
ax.semilogx(r_arr, save_ad, color=GREEN,  lw=2.0, marker='o', ms=6,
            label='Adaptive OW')
ax.axvline(params.r, color=RED, lw=1.5, ls='--', alpha=0.7,
           label=f'True $r^*$={params.r:.3f}')
ax.axhline(0, color=TEXT, lw=0.8, ls='--', alpha=0.4)
ax.set_xlabel('Initial guess $r_0$ (log scale)')
ax.set_ylabel('Cost savings vs TWAP (%)')
ax.set_title('Cost Savings vs TWAP')
ax.legend(fontsize=9)
ax.set_facecolor(GRID)

# Panel 2: adaptive advantage
ax = axes[1]
bar_colors = [GREEN if v >= 0 else RED for v in ad_beats]
ax.bar(range(len(r_vals)), ad_beats, color=bar_colors, alpha=0.8, edgecolor='none')
ax.axhline(0, color=TEXT, lw=1.0, ls='--')
ax.set_xticks(range(len(r_vals)))
ax.set_xticklabels([f'{r:.3f}' for r in r_vals], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Initial guess $r_0$')
ax.set_ylabel('Adaptive advantage (% savings over Static OW)')
ax.set_title('Adaptive OW − Static OW\n(green = adaptive wins)')
ax.set_facecolor(GRID)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb3_robustness.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()

print("\nDetailed results:")
print(f"{'r0':>8}  {'Static savings':>16}  {'Adaptive savings':>18}  {'Adaptive advantage':>20}")
print("-"*68)
for r0, ss, sa in zip(r_vals, save_st, save_ad):
    tag = " ← TRUE r" if abs(r0-params.r)<0.001 else ""
    print(f"{r0:>8.3f}  {ss:>14.4f}%  {sa:>16.4f}%  {sa-ss:>+18.4f}%{tag}")


## 3. Monte Carlo Distribution of Execution Costs

Full distribution of execution costs across 300 paths for three representative cases.


In [ ]:
r0_show = [0.01, params.r, 0.5]
labels  = [f'r₀={r}' for r in r0_show]

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=False)
fig.suptitle('Distribution of Execution Costs (300 MC paths)', color=TEXT, fontsize=12)

for i, (r0, label) in enumerate(zip(r0_show, labels)):
    ax  = axes[i]
    c_t = np.array(costs_tw_all)
    c_s = np.array(costs_st_dict[r0])
    c_a = np.array(costs_ad_dict[r0])

    bins = np.linspace(min(c_t.min(), c_s.min(), c_a.min()),
                       max(c_t.max(), c_s.max(), c_a.max()), 40)

    ax.hist(c_t, bins=bins, color=RED,    alpha=0.55, label=f'TWAP  μ=${c_t.mean():,.0f}')
    ax.hist(c_s, bins=bins, color=YELLOW, alpha=0.55, label=f'Static μ=${c_s.mean():,.0f}')
    ax.hist(c_a, bins=bins, color=GREEN,  alpha=0.55, label=f'Adaptive μ=${c_a.mean():,.0f}')

    ax.axvline(c_t.mean(), color=RED,    lw=1.5, ls='--')
    ax.axvline(c_s.mean(), color=YELLOW, lw=1.5, ls='--')
    ax.axvline(c_a.mean(), color=GREEN,  lw=1.5, ls='--')

    ax.set_xlabel('Total execution cost ($)')
    ax.set_ylabel('Count' if i == 0 else '')
    ax.set_title(f'Starting guess {label}', color=TEXT)
    ax.legend(fontsize=7)
    ax.set_facecolor(GRID)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb3_cost_dist.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()


## 4. Estimator Convergence Inside the Control Loop

The closed-loop coupling means that $\hat{r}_t$ is driven by a **non-stationary**
$D_t$ process (because the trades change $D_t$, and the trades depend on $\hat{r}_{t-}$).
Yet Theorem 4.3 still guarantees convergence — this is precisely the contribution
of the thesis: *no stationarity assumption is needed*.

Below we show the ensemble of $\hat{r}_t$ paths across 100 MC runs.


In [ ]:
n_show = 100
r0_fix = 2.0

r_hat_paths = np.zeros((n_show, T + 1))
for seed in range(n_show):
    env = OWEnvironment(params, X0=X0, T=T, dt=dt, seed=seed)
    res = env.run_adaptive_ow(r0=r0_fix, eta0=0.1, alpha=0.8)
    r_hat_paths[seed] = res.r_hat

t_arr  = np.arange(T + 1)
q10    = np.percentile(r_hat_paths, 10, axis=0)
q50    = np.percentile(r_hat_paths, 50, axis=0)
q90    = np.percentile(r_hat_paths, 90, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(
    f'$\hat{{r}}_t$ Convergence Inside the Adaptive Control Loop  '
    f'({n_show} MC paths, $r_0$={r0_fix})',
    color=TEXT, fontsize=12)

for ax, logy in zip(axes, [False, True]):
    for i in range(min(30, n_show)):
        ax.plot(t_arr, r_hat_paths[i], color=CYAN, alpha=0.15, lw=0.7)
    ax.fill_between(t_arr, q10, q90, color=BLUE, alpha=0.2,
                    label='10%–90% band')
    ax.plot(t_arr, q50, color=BLUE, lw=2.0, label='Median $\hat{r}_t$')
    ax.axhline(params.r, color=RED, lw=1.5, ls='--',
               label=f'True $r^*$ = {params.r:.4f}')
    ax.axhline(r0_fix, color=ORANGE, lw=1.0, ls=':', alpha=0.7,
               label=f'Initial $r_0$ = {r0_fix}')
    ax.set_xlabel('Time (seconds)')
    ax.set_ylabel('$\hat{r}_t$')
    ax.legend(fontsize=9)
    ax.set_facecolor(GRID)
    if logy:
        ax.set_yscale('log')
        ax.set_title('Log scale', color=TEXT)
    else:
        ax.set_title('Linear scale', color=TEXT)

plt.tight_layout()
plt.savefig(os.path.join('..','outputs','nb3_rhat_ensemble.png'),
            dpi=130, bbox_inches='tight', facecolor=DARK)
plt.show()

print(f"Median r_hat at t=0:    {q50[0]:.4f}")
print(f"Median r_hat at t=T/4:  {q50[T//4]:.4f}")
print(f"Median r_hat at t=T/2:  {q50[T//2]:.4f}")
print(f"Median r_hat at t=T:    {q50[-1]:.4f}")
print(f"True r*:                {params.r:.4f}")


## Summary

### What we showed

1. **The closed-loop system works as designed**: the estimator and controller
   operate simultaneously on the same time scale, with no artificial time-scale
   separation — matching the theoretical setup of the thesis.

2. **Adaptive OW beats Static OW under misspecification**: when the initial
   guess $r_0 < r^*$ (underestimating resilience — the common practical case),
   the adaptive strategy saves meaningful execution cost compared to a frozen
   static strategy.

3. **The estimator converges inside the control loop**: $\hat{r}_t \to r^*$
   across all MC paths, consistent with Theorem 4.3 (almost-sure convergence
   without stationarity or time-scale separation).

### Connection to the thesis

| Thesis result | Notebook demonstration |
|---------------|----------------------|
| Theorem 4.3: a.s. convergence | Panel 4: $\hat{r}_t$ ensemble convergence |
| Corollary 4.4: mean-square convergence | Panel 4: 10%–90% band narrows over time |
| Fig 5.1: estimator trajectories | Section 1: single-path $\hat{r}_t$ plot |
| Fig 5.3: error vs learning rate | Notebook 2, Section 2 |
| Closed-loop setting (no stationarity) | The simulation itself — no stationarity assumed |
